# Práctica — Preprocesamiento de Datos
## Data Preprocessing (Calidad, Limpieza, Integración, Reducción, Transformación/Discretización)

## Objetivo general
Que el estudiante implemente un flujo universal de **preprocesamiento de datos** en Python, seleccionando (comentando/descomentando) los pasos adecuados según:
- el **formato** del dataset (CSV/Excel/JSON/Parquet/SQL),
- el **tipo de atributos** (numéricos/categóricos/fechas/texto),
- la **calidad** de los datos (faltantes, ruido, outliers, duplicados),
- el **escenario** (integración de fuentes, reducción, discretización).

## Entregable (resumen)
- Dataset cargado y descripción básica.
- Evaluación de calidad (faltantes, tipos, inconsistencias).
- Limpieza aplicada (faltantes, ruido/outliers, formatos).
- (Opcional) Integración (si hay 2+ fuentes) y resolución de conflictos.
- Reducción (selección de atributos / PCA / muestreo, según aplique).
- Transformación y/o discretización (normalización, binning, etc.).
- Dataset final preprocesado exportado a archivo.


In [ ]:
# 0) Librerías base
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

# Opcionales
# !pip install scikit-learn openpyxl pyarrow fastparquet
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import StandardScaler, MinMaxScaler
# from sklearn.decomposition import PCA
# from sklearn.cluster import KMeans


## 1) Carga universal del dataset
Seleccione el **formato** correspondiente. Si su dataset está separado por `;` o usa codificación especial, ajuste los parámetros.

**ACTIVIDAD:** Indique en un comentario:
- nombre del archivo,
- formato,
- separador,
- codificación,
- columnas principales y objetivo (si existe).


In [ ]:
# 1.1 Ruta del archivo (ajuste)
DATA_PATH = "TU_ARCHIVO_AQUI"  # Ej: "datos.csv", "dataset.xlsx", "data.json", "data.parquet"

# 1.2 Seleccione el tipo de archivo (descomente solo UNA opción)

# --- Opción A: CSV ---
# df = pd.read_csv(DATA_PATH)

# --- CSV con separador y codificación ---
# df = pd.read_csv(DATA_PATH, sep=";", encoding="utf-8")

# --- Opción B: Excel (.xlsx) ---
# df = pd.read_excel(DATA_PATH, sheet_name=0)

# --- Opción C: JSON ---
# df = pd.read_json(DATA_PATH)

# --- Opción D: Parquet ---
# df = pd.read_parquet(DATA_PATH)

# --- Opción E: TXT delimitado ---
# df = pd.read_csv(DATA_PATH, sep="\t")

# --- Opción F: SQL (ejemplo con SQLite) ---
# import sqlite3
# conn = sqlite3.connect("mi_base.db")
# df = pd.read_sql_query("SELECT * FROM mi_tabla", conn)

# Verificación mínima
# display(df.head())
# print("Shape:", df.shape)


### 1.3 Si el dataset es muy grande (opcional)
- Cargar solo algunas columnas.
- Cargar una muestra con `nrows`.
- Leer por chunks.

Descomente si aplica.


In [ ]:
# --- Cargar solo columnas específicas ---
# cols = ["col1", "col2", "col3"]
# df = pd.read_csv(DATA_PATH, usecols=cols)

# --- Cargar solo N filas ---
# df = pd.read_csv(DATA_PATH, nrows=20000)

# --- Lectura por chunks (para archivos muy grandes) ---
# chunks = []
# for ch in pd.read_csv(DATA_PATH, chunksize=50000):
#     chunks.append(ch)
# df = pd.concat(chunks, ignore_index=True)


## 2) Vista rápida y perfil inicial (Data Quality)
En esta sección se detectan problemas comunes de calidad:
- tipos mal interpretados,
- valores faltantes,
- valores fuera de rango,
- categorías inconsistentes,
- duplicados.

**ACTIVIDAD:** Ejecute y describa 3 hallazgos importantes.


In [ ]:
# 2.1 Resumen rápido
display(df.head())
print("Filas, Columnas:", df.shape)
display(df.sample(min(5, len(df)), random_state=7))

# 2.2 Tipos y nulos
display(df.dtypes)
display(df.isna().sum().sort_values(ascending=False).head(20))

# 2.3 Estadística numérica
display(df.describe(include=[np.number]).T)

# 2.4 Estadística categórica
display(df.describe(include=["object", "category"]).T.head(20))


### 2.5 Detección de columnas de fecha/hora (opcional)
Si alguna columna representa una fecha pero aparece como texto, conviértala.

**ACTIVIDAD:** Identifique columnas de fecha y conviértalas si aplica.


In [ ]:
# Ejemplo: convertir una columna a datetime
# df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce", dayfirst=True)

# Validar conversiones
# display(df[["fecha"]].head())
# print(df["fecha"].isna().sum(), "fechas inválidas convertidas a NaT")


## 3) Data Cleaning — Valores faltantes (Missing Values)
Estrategias típicas:
- eliminar filas/columnas (si el daño es grande),
- imputar (media/mediana/moda),
- imputar por grupos,
- usar un valor "desconocido" en categóricas,
- interpolación (series temporales).

**ACTIVIDAD:** Elija una estrategia por tipo de variable y justifique.


In [ ]:
# 3.1 Identificar columnas por tipo
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numéricas:", num_cols[:20], "..." if len(num_cols)>20 else "")
print("Categóricas:", cat_cols[:20], "..." if len(cat_cols)>20 else "")


In [ ]:
# 3.2 Opción A: eliminar filas con nulos (simple, pero puede perder datos)
# df_clean = df.dropna().copy()

# 3.3 Opción B: eliminar columnas con demasiados nulos
# threshold = 0.5  # elimina columnas con >50% nulos
# to_drop = [c for c in df.columns if df[c].isna().mean() > threshold]
# df_clean = df.drop(columns=to_drop).copy()

# 3.4 Opción C: imputación simple
# df_clean = df.copy()

# --- numéricas: mediana ---
# for c in num_cols:
#     df_clean[c] = df_clean[c].fillna(df_clean[c].median())

# --- categóricas: moda o etiqueta ---
# for c in cat_cols:
#     if df_clean[c].isna().any():
#         moda = df_clean[c].mode(dropna=True)
#         if len(moda) > 0:
#             df_clean[c] = df_clean[c].fillna(moda.iloc[0])
#         else:
#             df_clean[c] = df_clean[c].fillna("DESCONOCIDO")

# 3.5 Opción D: imputar por grupo (ej. por una categoría)
# df_clean = df.copy()
# df_clean["col_num"] = df_clean.groupby("col_categoria")["col_num"].transform(lambda s: s.fillna(s.median()))

# Verificación
# print("Nulos después:", df_clean.isna().sum().sum())
# display(df_clean.head())


## 4) Data Cleaning — Ruido, outliers y formatos (Noisy Data)
Ruido puede venir de:
- valores extremos (outliers),
- errores de captura (negativos imposibles),
- unidades mezcladas,
- texto mal escrito (categorías).

Se recomiendan pasos:
1) identificar outliers,
2) decidir tratamiento (cap/winsorize, eliminar, transformar),
3) validar con contexto del problema.

**ACTIVIDAD:** Elija 1–3 columnas numéricas y aplique una técnica.


In [ ]:
# Seleccione columnas numéricas para revisar
cols_check = num_cols[:3]  # ACTIVIDAD: cambie a columnas relevantes
display(df[cols_check].describe().T)

# Visualización rápida (boxplot)
df[cols_check].boxplot()
plt.title("Boxplot (detección visual de outliers)")
plt.show()


In [ ]:
# 4.1 Outliers por IQR (ejemplo genérico)
def iqr_bounds(s: pd.Series, k: float = 1.5):
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return lower, upper

# Opción A: eliminar outliers
# df_clean2 = df_clean.copy()  # use df_clean si ya hizo imputación, si no, use df
# for c in cols_check:
#     lo, hi = iqr_bounds(df_clean2[c].dropna())
#     df_clean2 = df_clean2[(df_clean2[c] >= lo) & (df_clean2[c] <= hi)]
# print("Shape tras eliminar outliers:", df_clean2.shape)

# Opción B: cap/winsorize (recortar a límites)
# df_clean2 = df_clean.copy()
# for c in cols_check:
#     lo, hi = iqr_bounds(df_clean2[c].dropna())
#     df_clean2[c] = df_clean2[c].clip(lo, hi)
# display(df_clean2[cols_check].describe().T)


### 4.2 Limpieza de categorías inconsistentes (opcional)
Ejemplo: `"MEX"`, `"Mexico"`, `"México"` deben unificarse.

**ACTIVIDAD:** Elija 1 columna categórica y normalice (strip, lower, mapeo).


In [ ]:
# Seleccione una columna categórica a limpiar
# col_cat = cat_cols[0]

# Normalización básica (espacios y mayúsculas/minúsculas)
# df_clean[col_cat] = df_clean[col_cat].astype(str).str.strip().str.lower()

# Mapeo manual (ejemplo)
# mapa = {"méxico": "mexico", "mx": "mexico", "u.s.a": "usa", "united states": "usa"}
# df_clean[col_cat] = df_clean[col_cat].replace(mapa)

# Ver antes/después
# display(df[col_cat].value_counts().head(10))
# display(df_clean[col_cat].value_counts().head(10))


## 5) Data Integration — Unión de fuentes (opcional)
Esta sección aplica si hay **2 o más** datasets que deben integrarse.

Temas:
- Identificación de entidades (keys),
- Detección de duplicados de tuplas,
- Redundancia / correlación,
- Conflictos de valores (misma entidad, distinto valor).

**ACTIVIDAD:** Si NO integra, escriba: “No aplica: solo se usó una fuente”.


In [ ]:
# 5.1 Cargar una segunda fuente (opcional)
# df2 = pd.read_csv("OTRA_FUENTE.csv")

# 5.2 Unión por llave (merge)
# key = "id"  # ACTIVIDAD: cambie
# df_merged = df_clean.merge(df2, on=key, how="inner")  # inner/left/right/outer
# print(df_merged.shape)
# display(df_merged.head())


In [ ]:
# 5.3 Duplicación de tuplas (duplicados)
# base = df_clean  # o df_merged
# dup_rows = base.duplicated().sum()
# print("Duplicados exactos:", dup_rows)

# Eliminar duplicados exactos
# base_nodup = base.drop_duplicates().copy()
# print("Shape sin duplicados:", base_nodup.shape)


In [ ]:
# 5.4 Identificación de entidades: duplicados por llave
# base = df_clean
# key = "id"
# if key in base.columns:
#     print("IDs repetidos:", base[key].duplicated().sum())
#     # Mantener el registro más reciente (si hay fecha)
#     # base = base.sort_values("fecha").drop_duplicates(subset=[key], keep="last")


In [ ]:
# 5.5 Redundancia y correlación (numéricas)
# base = df_clean
# corr = base.select_dtypes(include=[np.number]).corr(numeric_only=True)
# display(corr)

# Visualización simple
# plt.imshow(corr.values)
# plt.title("Matriz de correlación (numéricas)")
# plt.colorbar()
# plt.show()


### 5.6 Conflictos de valores (opcional)
Ejemplo: misma entidad con distintos valores en 2 fuentes (teléfono, dirección, edad).
Estrategias:
- prioridad de una fuente,
- valor más reciente,
- regla de negocio (promedio, mayoría, etc.).

**ACTIVIDAD:** Si aplica, describa la regla usada.


In [ ]:
# Ejemplo conceptual (requiere llave e información duplicada):
# base_conf = df_merged.copy()
# # regla: si valor_x es nulo, tomar valor_x2; si no, conservar
# base_conf["valor_final"] = base_conf["valor_x"].combine_first(base_conf["valor_y"])


## 6) Data Reduction — Reducción de datos
Objetivo: disminuir tamaño o dimensionalidad conservando información relevante.

Estrategias típicas:
- **Selección de atributos** (quitar columnas irrelevantes o redundantes),
- **PCA** (reducción dimensional),
- **Histograms** (resumen),
- **Clustering** (representantes),
- **Sampling** (muestras representativas).

**ACTIVIDAD:** Aplique al menos **una** técnica y justifique.


In [ ]:
# 6.1 Selección de atributos (manual)
# base = df_clean
# keep = ["col1", "col2", "col3"]  # ACTIVIDAD: defina
# base_red = base[keep].copy()
# display(base_red.head())


In [ ]:
# 6.2 PCA (opcional, requiere scikit-learn y solo numéricas)
# from sklearn.preprocessing import StandardScaler
# from sklearn.decomposition import PCA

# base = df_clean
# X = base.select_dtypes(include=[np.number]).dropna().to_numpy()

# Xz = StandardScaler().fit_transform(X)
# pca = PCA(n_components=2)  # ACTIVIDAD: cambie componentes
# Z = pca.fit_transform(Xz)

# print("Varianza explicada:", pca.explained_variance_ratio_, "Total:", pca.explained_variance_ratio_.sum())

# plt.scatter(Z[:,0], Z[:,1], s=10)
# plt.title("PCA (2 componentes)")
# plt.xlabel("PC1"); plt.ylabel("PC2")
# plt.show()


In [ ]:
# 6.3 Sampling (muestreo)
# base = df_clean
# sample_frac = 0.1  # 10%
# base_sample = base.sample(frac=sample_frac, random_state=7)
# print("Muestra:", base_sample.shape)


In [ ]:
# 6.4 Clustering para resumen (opcional, requiere scikit-learn)
# from sklearn.cluster import KMeans
# from sklearn.preprocessing import StandardScaler

# base = df_clean
# X = base.select_dtypes(include=[np.number]).dropna()
# Xz = StandardScaler().fit_transform(X)

# k = 5  # ACTIVIDAD: elija k
# km = KMeans(n_clusters=k, n_init=10, random_state=7)
# labels = km.fit_predict(Xz)

# X_plot = Xz[:, :2] if Xz.shape[1] >= 2 else np.c_[Xz[:,0], np.zeros(len(Xz))]
# plt.scatter(X_plot[:,0], X_plot[:,1], c=labels, s=10)
# plt.title("Clustering (visualización 2D aproximada)")
# plt.show()


## 7) Data Transformation — Normalización y escalamiento
Transformaciones comunes:
- Min-Max (0 a 1),
- Z-score (centrar y escalar),
- Log (para colas largas),
- One-Hot encoding (categóricas) — si aplica al objetivo.

**ACTIVIDAD:** Elija una transformación para numéricas y una para categóricas (si aplica).


In [ ]:
# 7.1 Normalización manual (Min-Max y Z-score)
# base = df_clean
# cols = num_cols[:3]  # ACTIVIDAD

# base_t = base.copy()

# --- Min-Max ---
# for c in cols:
#     mn, mx = base_t[c].min(), base_t[c].max()
#     if mx != mn:
#         base_t[c] = (base_t[c] - mn) / (mx - mn)

# --- Z-score ---
# for c in cols:
#     mu, sd = base_t[c].mean(), base_t[c].std()
#     if sd != 0:
#         base_t[c] = (base_t[c] - mu) / sd

# display(base_t[cols].head())


In [ ]:
# 7.2 Transformación logarítmica (para colas largas / valores positivos)
# base = df_clean
# col = num_cols[0]  # ACTIVIDAD
# base_log = base.copy()

# # log1p permite log(1+x) y soporta ceros
# base_log[col] = np.log1p(base_log[col].clip(lower=0))
# display(base_log[[col]].describe())


In [ ]:
# 7.3 One-Hot Encoding (categóricas) — opcional
# base = df_clean
# cols = cat_cols[:2]  # ACTIVIDAD
# base_ohe = pd.get_dummies(base, columns=cols, drop_first=False)
# print("Antes:", base.shape, "Después:", base_ohe.shape)


## 8) Data Discretization — Binning y discretización por histograma
Discretizar significa convertir una variable numérica en categorías.

Técnicas:
- **Binning** por intervalos iguales (equal-width),
- Binning por cuantiles (equal-frequency),
- Discretización guiada por histograma (bins informados).

**ACTIVIDAD:** Elija una variable y cree una columna discretizada.


In [ ]:
# 8.1 Discretización por intervalos iguales (equal-width)
# base = df_clean
# col = num_cols[0]  # ACTIVIDAD
# k = 5  # número de bins

# base_disc = base.copy()
# base_disc[col + "_bin_w"] = pd.cut(base_disc[col], bins=k)
# display(base_disc[[col, col + "_bin_w"]].head())
# display(base_disc[col + "_bin_w"].value_counts())


In [ ]:
# 8.2 Discretización por cuantiles (equal-frequency)
# base = df_clean
# col = num_cols[0]  # ACTIVIDAD
# k = 5

# base_disc = base.copy()
# base_disc[col + "_bin_q"] = pd.qcut(base_disc[col], q=k, duplicates="drop")
# display(base_disc[[col, col + "_bin_q"]].head())
# display(base_disc[col + "_bin_q"].value_counts())


## 9) Discretización por clustering / árbol / correlación (opcional)
Estas técnicas son más avanzadas y suelen depender del objetivo (clasificación/regresión).

- **Clustering**: formar grupos y usar el cluster como categoría.
- **Árbol de decisión**: cortes que maximizan ganancia de información.
- **Correlación**: discretizar para mejorar relación con variable objetivo.

**ACTIVIDAD:** Si su dataset tiene variable objetivo, intente al menos una.


In [ ]:
# 9.1 Discretización por clustering (requiere scikit-learn)
# from sklearn.cluster import KMeans
# from sklearn.preprocessing import StandardScaler

# base = df_clean
# col = num_cols[0]  # ACTIVIDAD
# X = base[[col]].dropna().to_numpy()

# Xz = StandardScaler().fit_transform(X)
# km = KMeans(n_clusters=4, n_init=10, random_state=7)
# clusters = km.fit_predict(Xz)

# base_cluster = base.dropna(subset=[col]).copy()
# base_cluster[col + "_cluster"] = clusters
# display(base_cluster[[col, col + "_cluster"]].head())


## 10) Concept Hierarchy Generation (Nominal Data) — opcional
Una jerarquía de conceptos agrupa categorías en niveles.

Ejemplo:
- `Ciudad` → `Estado` → `País`

**ACTIVIDAD:** Si su dataset tiene una columna nominal con demasiadas categorías, agrúpelas en una jerarquía simple.


In [ ]:
# Ejemplo: agrupar categorías raras en "OTROS"
# base = df_clean
# col = cat_cols[0]  # ACTIVIDAD
# top_n = 10

# vc = base[col].value_counts()
# top = set(vc.head(top_n).index)

# base_h = base.copy()
# base_h[col + "_hier"] = base_h[col].where(base_h[col].isin(top), other="OTROS")
# display(base_h[[col, col + "_hier"]].head())
# display(base_h[col + "_hier"].value_counts())


## 11) Exportación del dataset final
**ACTIVIDAD:** Exporte el dataset preprocesado final.

Sugerencias:
- CSV para interoperabilidad,
- Parquet si es grande (mejor compresión),
- Excel si se requiere reporte.

Recuerde guardar con un nombre que indique el estado, por ejemplo:
`dataset_preprocesado_v1.csv`.


In [ ]:
# Defina cuál es su dataset final:
# final_df = df_clean2  # ejemplo: después de outliers
# final_df = df_clean   # ejemplo: después de imputación

# Exportación
# final_df.to_csv("dataset_preprocesado_v1.csv", index=False)
# final_df.to_parquet("dataset_preprocesado_v1.parquet", index=False)
# final_df.to_excel("dataset_preprocesado_v1.xlsx", index=False)

# print("Exportación lista.")


## 12) Contestar (como parte de la conclusión)
**ACTIVIDAD:** Redacte un resumen final:
1. ¿Qué problemas de calidad detectó?
2. ¿Qué técnicas aplicó y por qué?
3. ¿Qué cambió en el dataset (filas/columnas/rangos)?
4. ¿Qué riesgos quedan (posibles sesgos, pérdida de información)?
